In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_country = pd.read_csv("../data/raw/GlobalLandTemperaturesByCountry.csv")
df_global = pd.read_csv("../data/raw/Dataset1.csv")

In [ ]:
df_country['dt'] = pd.to_datetime(df_country['dt'])
df_country['Year'] = df_country['dt'].dt.year
df_country['Month'] = df_country['dt'].dt.month
df_country['Country'] = df_country['Country'].astype(str).str.strip()

df_global = df_global.rename(columns={'year': 'Year', 'country': 'Country'})
df_global['Country'] = df_global['Country'].astype(str).str.strip()
df_global['Year'] = pd.to_numeric(df_global['Year'], errors='coerce')
df_country['Year'] = pd.to_numeric(df_country['Year'], errors='coerce')
df_country['AverageTemperature'] = pd.to_numeric(df_country['AverageTemperature'], errors='coerce')
df_country['AverageTemperatureUncertainty'] = pd.to_numeric(df_country['AverageTemperatureUncertainty'], errors='coerce')
value_cols = [c for c in df_global.columns if c not in ['Year', 'Country']]
df_global = df_global.groupby(['Year', 'Country'], as_index=False)[value_cols].mean(numeric_only=True)

df_country_yearly = df_country.groupby(['Year', 'Country'], as_index=False).agg(
    AverageTemperature=('AverageTemperature', 'mean'),
    AverageTemperatureUncertainty=('AverageTemperatureUncertainty', 'mean')
)

In [ ]:
df_merged = pd.merge(
    df_country_yearly,
    df_global,
    on=['Year', 'Country'],
    how='inner',
    validate='one_to_one'
)

In [ ]:
feature_cols = [c for c in df_global.columns if c not in ['Year', 'Country']]
df_model = df_merged.dropna(subset=['AverageTemperature'] + feature_cols)
df_model = df_model.sort_values(['Country', 'Year']).reset_index(drop=True)

In [ ]:
df_model.to_csv("../data/processed/model_dataset.csv", index=False)